In [1]:
import glob
import re
import numpy as np
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from pathlib import Path
from typing import List

In [2]:


def collect_files(base_dir: str, dirs: List[str], subpath: str, pattern: str, exclude: str = None) -> List[str]:
    """
    Collects files matching a pattern from specified subdirectories.

    Parameters:
        base_dir (str): Base directory containing all subdirectories.
        dirs (List[str]): List of subdirectory names.
        subpath (str): Relative path inside each subdirectory where files are located.
        pattern (str): Glob pattern to match files.
        exclude (str, optional): String pattern to exclude from results.

    Returns:
        List[str]: Sorted list of matching file paths as strings.
    """
    all_files = []
    for d in dirs:
        search_path = Path(base_dir) / d / subpath / pattern
        files = sorted(search_path.parent.glob(search_path.name))
        if exclude:
            files = [f for f in files if exclude not in f.name]
        all_files.extend([str(f) for f in files])
    return all_files

# Base directory and subdirectories
base_dir = "/home/jovyan/SAMBBA_data/dap.ceda.ac.uk/badc/faam/data/2012"
dirs = [
    "b732-sep-15", "b733-sep-16", 
    "b734-sep-18", "b735-sep-19", "b736-sep-19", "b737-sep-20", "b738-sep-22", 
    "b739-sep-23", "b740-sep-25", "b741-sep-26", "b742-sep-27", "b743-sep-27", 
    "b744-sep-28", "b745-sep-28", "b746-sep-29", "b747-oct-01", "b748-oct-02", 
    "b749-oct-03", "b750-oct-03"
]

# Collect files
core_processed_files = collect_files(base_dir, dirs, "core_processed", "core_faam*r1*.nc", exclude="_1hz")
dry_neph_files = collect_files(base_dir, dirs, "mo-non-core", "metoffice*neph1*.nc")
flight_sum_files = collect_files(base_dir, dirs, ".", "flight-sum*.txt")
sp2_files = collect_files(base_dir, dirs, "non-core", "man-sp2*.na")
ams_files = collect_files(base_dir, dirs, "non-core", "man-ams*.na")

# Example: print or pass them into other analysis
# print(core_processed_files)


In [3]:
# Function to extract flight number (e.g., "b734") from a file path
def extract_flight_number(file_path):
    match = re.search(r'b\d{3}', file_path)
    return match.group(0) if match else None  # Returns None if no match

# Get flight numbers for each dataset
flights_nc = {extract_flight_number(f) for f in core_processed_files}
flights_ams = {extract_flight_number(f) for f in dry_neph_files}
flights_sp2 = {extract_flight_number(f) for f in sp2_files}
flights_neph = {extract_flight_number(f) for f in ams_files}
flights_sum = {extract_flight_number(f) for f in flight_sum_files}

# Find flights that are in all three datasets
common_flights = flights_nc & flights_ams & flights_sp2 & flights_neph & flights_sum # Set intersection

# Filter the file lists to include only matching flights
filtered_nc = [f for f in core_processed_files if extract_flight_number(f) in common_flights]
filtered_ams = [f for f in ams_files if extract_flight_number(f) in common_flights]
filtered_sp2 = [f for f in sp2_files if extract_flight_number(f) in common_flights]
filtered_neph = [f for f in dry_neph_files if extract_flight_number(f) in common_flights]
filtered_sum = [f for f in flight_sum_files if extract_flight_number(f) in common_flights]

# Print the results
print(f"Common flights: {sorted(common_flights)}")
print(f"Filtered NC files: {len(filtered_nc)}")
print(f"Filtered AMS files: {len(filtered_ams)}")
print(f"Filtered SP2 files: {len(filtered_sp2)}")
print(f"Filtered Neph files: {len(filtered_sp2)}")
print(f"Filtered Flight sum files: {len(filtered_sum)}")

Common flights: ['b734', 'b737', 'b739', 'b740', 'b741', 'b742', 'b743', 'b744', 'b745', 'b746', 'b747', 'b748', 'b749', 'b750']
Filtered NC files: 14
Filtered AMS files: 14
Filtered SP2 files: 14
Filtered Neph files: 14
Filtered Flight sum files: 14


In [4]:
filtered_sp2[0]

# Extract the date (format: yyyymmdd)
match = re.search(r'(\d{8})', filtered_sp2[0])

# Get the matched date or None if not found
flight_date = match.group(1) if match else None

print(flight_date)  # Output: 20120914

20120918


In [5]:
def convert_time_to_datetime_sp2(dsf, start_date_str):
    """
    Convert the 'Time_start_UTC' variable (seconds from 00:00 UTC) into datetime format.
    
    Parameters:
    - ds (xarray.Dataset): The dataset containing 'Time_start_UTC'
    - start_date_str (str): Start date in "YYYY MM DD" format
    
    Returns:
    - xarray.Dataset: Dataset with a new 'datetime' coordinate
    """
    # Convert start_date string to datetime object
    start_date = datetime.strptime(start_date_str, "%Y%m%d")
    
    # Extract 'Time_start_UTC' variable (seconds from 00:00 UTC)
    time_seconds = dsf["Time_UTC"].values
    
    # Convert seconds to actual datetime
    datetime_values = np.array([start_date + timedelta(seconds=int(sec)) for sec in time_seconds])

    # Assign datetime as a coordinate in the dataset
    dsf = dsf.assign_coords(datetime=("time", datetime_values))

    return dsf
    
def convert_time_to_datetime_ams(dsf, start_date_str):
    """
    Convert the 'Time_start_UTC' variable (seconds from 00:00 UTC) into datetime format.
    
    Parameters:
    - ds (xarray.Dataset): The dataset containing 'Time_start_UTC'
    - start_date_str (str): Start date in "YYYY MM DD" format
    
    Returns:
    - xarray.Dataset: Dataset with a new 'datetime' coordinate
    """
    # Convert start_date string to datetime object
    start_date = datetime.strptime(start_date_str, "%Y%m%d")
    
    # Extract 'Time_start_UTC' variable (seconds from 00:00 UTC)
    time_seconds = dsf["Time_start_UTC"].values
    
    # Convert seconds to actual datetime
    datetime_values = np.array([start_date + timedelta(seconds=int(sec)) for sec in time_seconds])

    # Assign datetime as a coordinate in the dataset
    dsf = dsf.assign_coords(datetime=("time", datetime_values))

    return dsf
    
def convert_time_to_datetime_neph(dsf, start_date_str):
    
    # Convert start_date string to datetime object
    start_date = datetime.strptime(start_date_str, "%Y%m%d")

    key_options = ["neph_spm", "NEPH_SPM"]  # Possible variations of the key
    matching_key = next((key for key in key_options if key in dsf), None)
    
    if matching_key:
        time_seconds = dsf[matching_key].values
    else:
        raise KeyError("Variable 'neph_spm' not found in dataset.")
    
    # Extract 'Time_start_UTC' variable (seconds from 00:00 UTC)
    #time_seconds = dsf["neph_spm"].values
    
    # Convert seconds to actual datetime
    datetime_values = np.array([start_date + timedelta(seconds=int(sec)) for sec in time_seconds])

    # Assign datetime as a coordinate in the dataset
    dsf = dsf.assign_coords(datetime=("datetime", datetime_values))

    return dsf

In [6]:
def dataset_sp2_ams_neph(file_path_sp2,file_path_ams,file_path_neph):

    with open(file_path_sp2, "rb") as f:
        lines_sp2 = f.readlines()  # Read the first 100 bytes
    with open(file_path_ams, "rb") as f:
        lines_ams = f.readlines()  # Read the first 100 bytes
        
    metadata_lines_sp2 = lines_sp2[:39]  # 39 lines are metadata
    header_line_sp2 = metadata_lines_sp2[38].strip().split()  # Extract header names
    # Decode if header contains byte strings
    header_line_sp2 = [h.decode("utf-8") if isinstance(h, bytes) else h for h in header_line_sp2]
    # Decode if metadata contains byte strings
    metadata_lines_sp2 = [line.decode("utf-8").strip() if isinstance(line, bytes) else line.strip() for line in metadata_lines_sp2]
    #print(header_line_sp2)

    metadata_lines_ams = lines_ams[:46]   # 46 lines are metadata
    header_line_ams = metadata_lines_ams[45].strip().split()  # Extract header names
    # Decode if header contains byte strings
    header_line_ams = [h.decode("utf-8") if isinstance(h, bytes) else h for h in header_line_ams]
    # Decode if metadata contains byte strings
    metadata_lines_ams = [line.decode("utf-8").strip() if isinstance(line, bytes) else line.strip() for line in metadata_lines_ams]
    #print(header_line_ams)

    data_lines_sp2 = lines_sp2[39:]  # Actual data starts after metadata
    data_lines_ams = lines_ams[46:]
    
    # Parse data into a NumPy array
    data_array = np.array([line.strip().split() for line in data_lines_sp2], dtype=float)
    
    # Define dimensions (assuming first column is time and others are variables)
    dims = ("time", "variable")  # Modify as needed
    time_coords = np.arange(data_array.shape[0])  # Placeholder for time index
    
    # Convert to xarray Dataset with headers as variable names
    ds0_sp2 = xr.Dataset(
        {name: ("time", data_array[:, i]) for i, name in enumerate(header_line_sp2)},
        coords={"time": time_coords},
        attrs={"Metadata": "\n".join(metadata_lines_sp2[:38])}  # Store metadata as an attribute
    )
    
    # Parse data into a NumPy array
    data_array = np.array([line.strip().split() for line in data_lines_ams], dtype=float)
    
    # Define dimensions (assuming first column is time and others are variables)
    dims = ("time", "variable")  # Modify as needed
    time_coords = np.arange(data_array.shape[0])  # Placeholder for time index
    
    # Convert to xarray Dataset with headers as variable names
    ds0_ams = xr.Dataset(
        {name: ("time", data_array[:, i]) for i, name in enumerate(header_line_ams)},
        coords={"time": time_coords},
        attrs={"Metadata": "\n".join(metadata_lines_ams[:45])}  # Store metadata as an attribute
    )
 
    # Extract the date (format: yyyymmdd)
    match_sp2 = re.search(r'(\d{8})', file_path_sp2)
    # Get the matched date or None if not found
    flight_date_sp2 = match_sp2.group(1) if match_sp2 else None

    # Extract the date (format: yyyymmdd)
    match_ams = re.search(r'(\d{8})', file_path_ams)
    # Get the matched date or None if not found
    flight_date_ams = match_ams.group(1) if match_ams else None


    ds_neph = xr.open_dataset(file_path_neph,decode_times=False)

    # Extract the date (format: yyyymmdd)
    match_neph = re.search(r'(\d{8})', file_path_neph)
    # Get the matched date or None if not found
    flight_date_neph = match_neph.group(1) if match_neph else None
    
    # Convert time
    ds0_sp2 = convert_time_to_datetime_sp2(ds0_sp2, flight_date_sp2)
    ds0_ams = convert_time_to_datetime_ams(ds0_ams, flight_date_ams)
    ds_neph = convert_time_to_datetime_neph(ds_neph, flight_date_neph)
    
    #TM,AD= calculate_total_mass(ds_sp2,ds_ams)

    return ds0_sp2,ds0_ams,ds_neph

In [7]:
def calculate_absortion_eff(aux_ds_sp2,aux_ds_ams,aux_ds_psap):
    # Filter out invalid BC mass values (9999999)
    valid_BC_mass_mask = aux_ds_sp2["BC_mass"] != 9999999
    filtered_ds_BC_mass = aux_ds_sp2.where(valid_BC_mass_mask, drop=True)
    
    # Filter out invalid SO4 values (9999999)
    valid_SO4_mask = aux_ds_ams["SO4"] != 9999999
    filtered_ds_SO4 = aux_ds_ams.where(valid_SO4_mask, drop=True)
    
    # Filter out invalid NH4 values (9999999)
    valid_NH4_mask = aux_ds_ams["NH4"] != 9999999
    filtered_ds_NH4 = aux_ds_ams.where(valid_NH4_mask, drop=True)
    
    # Filter out invalid ORG values (9999999)
    valid_ORG_mask = aux_ds_ams["ORG"] != 9999999
    filtered_ds_ORG = aux_ds_ams.where(valid_ORG_mask, drop=True)
    
    # Filter out invalid NO3 values (9999999)
    valid_NO3_mask = aux_ds_ams["NO3"] != 9999999
    filtered_ds_NO3 = aux_ds_ams.where(valid_NO3_mask, drop=True)
    
    
    # Filter out invalid Chl values (9999999)
    valid_Chl_mask = aux_ds_ams["Chl"] != 9999999
    filtered_ds_Chl = aux_ds_ams.where(valid_Chl_mask, drop=True)
    
    # Convert BC_mass from ng/m³ to µg/m³ (1 ng = 0.001 µg)
    filtered_ds_BC_mass["BC_mass_ug"] = filtered_ds_BC_mass["BC_mass"] * 1e-3
    
    # Interpolate BC_mass_ug to match the timestamps of filtered_ds_SO4
    bc_mass_interp = np.interp(
        filtered_ds_SO4.datetime.astype('int64'),   # Target time (SO4)
        filtered_ds_BC_mass.datetime.astype('int64'), # Original BC_mass time
        filtered_ds_BC_mass["BC_mass_ug"] # Original BC_mass values
    )

    # Compute total mass concentration
    total_mass = (
        bc_mass_interp + 
        filtered_ds_SO4["SO4"] +
        filtered_ds_NH4["NH4"] +
        filtered_ds_ORG["ORG"] +
        filtered_ds_NO3["NO3"] +
        filtered_ds_Chl["Chl"]
    )
    
    filtered_time = total_mass.datetime.values  # Reference timestamps
    psap_time = aux_ds_psap["Time"].values  # Original timestamps from new dataset
    
    common_times = np.intersect1d(filtered_time, psap_time)

    # Create a boolean mask as an Xarray DataArray
    mask = xr.DataArray(np.isin(psap_time, common_times), dims=["data_point"])
    mask1 = xr.DataArray(np.isin(filtered_time, common_times), dims=["time"])
    
    # Apply the mask to filter ds_psap
    ds_psap_matched = aux_ds_psap.where(mask, drop=True)
    total_mass_matched = total_mass.where(mask1, drop=True)

    
    return ds_psap_matched,total_mass_matched

In [8]:
def Area_plot(aux_ds_sp2,aux_ds_ams,aux_ds_psap):
    # Filter out invalid BC mass values (9999999)
    valid_BC_mass_mask = aux_ds_sp2["BC_mass"] != 9999999
    filtered_ds_BC_mass = aux_ds_sp2.where(valid_BC_mass_mask, drop=True)
    
    # Filter out invalid SO4 values (9999999)
    valid_SO4_mask = aux_ds_ams["SO4"] != 9999999
    filtered_ds_SO4 = aux_ds_ams.where(valid_SO4_mask, drop=True)
    
    # Filter out invalid NH4 values (9999999)
    valid_NH4_mask = aux_ds_ams["NH4"] != 9999999
    filtered_ds_NH4 = aux_ds_ams.where(valid_NH4_mask, drop=True)
    
    # Filter out invalid ORG values (9999999)
    valid_ORG_mask = aux_ds_ams["ORG"] != 9999999
    filtered_ds_ORG = aux_ds_ams.where(valid_ORG_mask, drop=True)
    
    # Filter out invalid NO3 values (9999999)
    valid_NO3_mask = aux_ds_ams["NO3"] != 9999999
    filtered_ds_NO3 = aux_ds_ams.where(valid_NO3_mask, drop=True)
    
    
    # Filter out invalid Chl values (9999999)
    valid_Chl_mask = aux_ds_ams["Chl"] != 9999999
    filtered_ds_Chl = aux_ds_ams.where(valid_Chl_mask, drop=True)
    
    # Convert BC_mass from ng/m³ to µg/m³ (1 ng = 0.001 µg)
    filtered_ds_BC_mass["BC_mass_ug"] = filtered_ds_BC_mass["BC_mass"] * 1e-3
    
    # Interpolate BC_mass_ug to match the timestamps of filtered_ds_SO4
    bc_mass_interp = np.interp(
        filtered_ds_SO4.datetime.astype('int64'),   # Target time (SO4)
        filtered_ds_BC_mass.datetime.astype('int64'), # Original BC_mass time
        filtered_ds_BC_mass["BC_mass_ug"] # Original BC_mass values
    )

    # Compute total mass concentration
    total_mass = (
        bc_mass_interp + 
        filtered_ds_SO4["SO4"] +
        filtered_ds_NH4["NH4"] +
        filtered_ds_ORG["ORG"] +
        filtered_ds_NO3["NO3"] +
        filtered_ds_Chl["Chl"]
    )
    
    # Define time axis (assuming all datasets have the same time dimension)
    time = total_mass.datetime.values
    
    # Extract variables
    bc_mass = bc_mass_interp
    so4 = filtered_ds_SO4["SO4"]
    nh4 = filtered_ds_NH4["NH4"]
    org = filtered_ds_ORG["ORG"]
    no3 = filtered_ds_NO3["NO3"]
    chl = filtered_ds_Chl["Chl"]

    
    #filtered_time = total_mass.datetime.values  # Reference timestamps
    #psap_time = aux_ds_psap["Time"].values  # Original timestamps from new dataset
    
    # Stack the values
    data = [bc_mass, so4, nh4, org, no3, chl]
    
    return time,data


In [9]:
def match_flight_with_blh(ds_height, ds, resolution=0.25):
    """
    Match aircraft flight track data with boundary layer height (BLH) from reanalysis dataset.
    
    Parameters:
        ds_height (xarray.Dataset): Aircraft dataset with variables 'Time', 'LAT_GPS', 'LON_GPS', 'GPS_ALT'.
        ds (xarray.Dataset): Reanalysis dataset with 'valid_time', 'latitude', 'longitude', and 'blh'.
        resolution (float): Grid resolution for reanalysis data (default: 0.25 degrees).
        
    Returns:
        pd.DataFrame: DataFrame with aircraft time, lat/lon, altitude, and matched BLH values.
    """
    def round_to_grid(value, resolution=0.25):
        return np.round(value / resolution) * resolution

    # Extract and round aircraft data
    flight_time = ds_height['Time']
    flight_lat = round_to_grid(ds_height['LAT_GPS'], resolution)
    flight_lon = round_to_grid(ds_height['LON_GPS'], resolution)
    flight_hgt = ds_height['GPS_ALT']

    # Round time to the nearest hour
    flight_time_rounded = pd.to_datetime(flight_time.values).floor('h')

    # Reanalysis grid values
    rean_lat = ds['latitude'].values
    rean_lon = ds['longitude'].values
    rean_time = pd.to_datetime(ds['valid_time'].values)

    # Create aircraft dataframe
    df_aircraft = pd.DataFrame({
        'time': pd.to_datetime(flight_time.values),
        'time_hour': flight_time_rounded,
        'lat': flight_lat.values.squeeze(),
        'lon': flight_lon.values.squeeze(),
        'height': flight_hgt.values.squeeze()
    })

    # Lookup BLH for each aircraft point
    def get_blh(row):
        try:
            lat_idx = np.abs(rean_lat - row['lat']).argmin()
            lon_idx = np.abs(rean_lon - row['lon']).argmin()
            time_idx = np.where(rean_time == row['time_hour'])[0]
            if len(time_idx) == 0:
                return np.nan
            time_idx = time_idx[0]
            return ds['blh'].isel(valid_time=time_idx, latitude=lat_idx, longitude=lon_idx).item()
        except Exception as e:
            return np.nan

    df_aircraft['blh'] = df_aircraft.apply(get_blh, axis=1)

    return df_aircraft

In [10]:
ds_bdl = xr.open_dataset('../Boundary_Layer_REA_Data.nc')
ds_bdl['valid_time'] = ds_bdl['valid_time'] - pd.Timedelta(hours=3)

FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/SAMBBA-data-analysis/Scripts/Boundary_Layer_REA_Data.nc'

In [ ]:
ds_sp2,ds_ams,ds_neph = dataset_sp2_ams_neph(filtered_sp2[0],filtered_ams[0],filtered_neph[0])

In [ ]:
ds_psap = xr.open_dataset(filtered_nc[0])
minlon, maxlon, minlat, maxlat = -70, -55, -16, -2  # Example box 1
#valid_PSAP_mask = ds_psap["PSAP_LIN_FLAG"] == 0
#filtered_ds_psap = ds_psap.where(valid_PSAP_mask, drop=False)
ds_test,tm_test=calculate_absortion_eff(ds_sp2,ds_ams,ds_psap)

In [ ]:
def parse_flight_log(lines):
    extracted_data = []
    
    for line in lines[2:]:  # Skip headers
        parts = re.split(r'\s{2,}', line.strip())  # Split on 2+ spaces
        
        if len(parts) < 2:
            continue  # Skip lines with insufficient data
        
        start_time = parts[0]  # First column is always Start Time

        # Check if second column is numeric (End Time), if so, event is at index 2
        if len(parts) > 2 and parts[1].isdigit():
            event = parts[2]  # Third column is the event
        else:
            event = parts[1]  # Otherwise, second column is the event

        extracted_data.append([start_time, event])  # Append as list (not tuple)
        
    return np.array(extracted_data)  # Convert to NumPy array

In [ ]:
def flight_comments_df(file_path_flight_comm):
    # Read the file
    with open(file_path_flight_comm, "r") as file:
        lines = file.readlines()
    # Process and print output
    parsed_data = parse_flight_log(lines)
    
    start_time_col = parsed_data[3:, 0]  # First column
    event_col = parsed_data[3:, 1]  # Second column

    # Create DataFrame
    df_flight = pd.DataFrame({"Start Time": start_time_col, "Event": event_col})

    
    return df_flight

In [ ]:
# Read the file
with open(filtered_sum[0], "r") as file:
    lines = file.readlines()
# Process and print output
parsed_data = parse_flight_log(lines)

start_time_col = parsed_data[3:, 0]  # First column
event_col = parsed_data[3:, 1]  # Second column

# Create DataFrame
df_flight = pd.DataFrame({"Start Time": start_time_col, "Event": event_col})

# Print the DataFrame
#print(df_flight)

In [ ]:
# Assuming df is your DataFrame and has columns "Start Time" and "Event"
start_up_time = df_flight[df_flight["Event"] == "Start-Up"]["Start Time"].iloc[0]

# Print the extracted start time
print(start_up_time)

In [ ]:
# Step 1: Convert 'Start Time' from HHMMSS to datetime
df_flight["Start Time"] = pd.to_datetime(df_flight["Start Time"], format="%H%M%S")  # Convert HHMMSS to time only
df_flight["Start Time"] = df_flight["Start Time"].apply(lambda t: pd.Timestamp("2012-09-18") + pd.Timedelta(hours=t.hour, minutes=t.minute, seconds=t.second))

# Extract time values from the dataset (since it's a variable, not a dimension)
ds_times = ds_psap["Time"].values  # Convert to NumPy array

# Step 2: Plot the height over time
plt.figure(figsize=(10, 5))

# Plot height (HGT_RADR) over time
plt.plot(ds_times, ds_psap["HGT_RADR"].values, color="b", linestyle="-", alpha=0.7, label="Height")

# Step 3: Overlay event labels
for event_time, event_name in zip(df_flight["Start Time"], df_flight["Event"]):
    # Find the closest time in the dataset
    closest_idx = np.argmin(np.abs(ds_times - np.datetime64(event_time)))  # Index of nearest timestamp
    closest_time = ds_times[closest_idx]
    height_at_time = ds_psap["HGT_RADR"].values[closest_idx][0]  # Get height value

    # Plot event text
    plt.text(closest_time, height_at_time, event_name, fontsize=6, color="red", ha="left", va="bottom", rotation=45)


# Assuming df is your DataFrame and has columns "Start Time" and "Event"
start_up_time = df_flight[df_flight["Event"] == "Start-Up"]["Start Time"].iloc[0]

# Set xlim to start at 11:40
plt.xlim(start_up_time, ds_times[-1])  # ds_times[-1] is the last time value

# Labels and formatting
plt.xlabel("Time")
plt.ylabel(r"Height (m)")
plt.xticks(rotation=45)
plt.grid()
#plt.legend()

# Tight layout for better plot formatting
#plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
area_time,data_mass_concentration = Area_plot(ds_sp2,ds_ams,ds_psap)
data_mass_concentration

In [ ]:

#labels = ["BC", "SO4", "NH4", "ORG", "NO3", "Chl"]
#colors = ["black", "blue", "green", "orange", "red", "purple"]  # Assign colors
labels = ["BC", "SO4", "NH4", "ORG", "NO3", "Chl"]
colors = ["black", "lightblue", "purple", "brown", "red", "green"]
# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.stackplot(area_time, data_mass_concentration, labels=labels, colors=colors, alpha=0.7)

# Labels and legend
ax.set_xlabel("Time")
ax.set_ylabel("Mass Concentration (µg/m³)")
ax.set_title("Stacked Area Plot of Mass Concentrations")
ax.legend(loc="upper left")

plt.show()

In [ ]:
# Create a figure and axis with a Cartopy map projection
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()})

# Filter valid points (assuming 1 means valid, adjust this logic based on your flag meaning)
valid_points_lat = ds_psap['LAT_GIN_FLAG'].values == 1
valid_points_lon = ds_psap['LON_GIN_FLAG'].values == 1
# Select only valid latitude and longitude points using numpy boolean indexing
lat_valid = ds_psap['LAT_GIN'].values[valid_points_lat]
lon_valid = ds_psap['LON_GIN'].values[valid_points_lon]

# Set map extent (adjust based on the region you're plotting)
ax.set_extent([minlon, maxlon, minlat, maxlat], crs=ccrs.PlateCarree())

# Add map features
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.gridlines(draw_labels=True)

# Loop through each dataset to plot flight tracks
#for file in (file_paths):
#ds = xr.open_dataset(file)

# Extract latitude and longitude for the current dataset
lat = ds_psap['LAT_GIN'].values.flatten()
lon = ds_psap['LON_GIN'].values.flatten()

# Get the title for the current dataset to use as a label
track_title = ds_psap.attrs.get('title', 'Unknown Track')

# Plot the latitude and longitude points for the current flight track
ax.scatter(lon, lat, s=1, transform=ccrs.PlateCarree())

# Optionally add a title and legend
plt.title('Flight Track\n'+track_title)
#plt.legend(loc='upper right')

# Show the plot
plt.show()

In [ ]:
valid_PSAP_mask = ds_test["PSAP_LIN_FLAG"] == 0
filtered_ds_psap = ds_test.where(valid_PSAP_mask, drop=False)
filtered_ds_psap_aux = ds_test.where(valid_PSAP_mask, drop=True)
# Create a boolean mask as an Xarray DataArray
# Ensure both arrays are 1D and aligned
psap_values = filtered_ds_psap["PSAP_LIN"].values.squeeze()
tm_values = tm_test.squeeze()

# Perform element-wise division, ignoring NaNs
result = np.divide(psap_values, tm_values*1e-6)

# Drop NaN values from the result if needed
Mass_absortion_eff = result.where(~np.isnan(result), drop=True)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))  # 2 rows, 2 columns
formatter = mticker.ScalarFormatter(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((-2, 2))  # Force scientific notation when appropriate

# First plot - Stacked Area Plot
labels = ["BC", "SO4", "NH4", "ORG", "NO3", "Chl"]
colors = ["black", "blue", "green", "orange", "red", "purple"]
axes[0, 0].stackplot(area_time, data_mass_concentration, labels=labels, colors=colors, alpha=0.7)
axes[0, 0].set_xlabel("Time")
axes[0, 0].set_ylabel("Mass Concentration (µg/m³)")
axes[0, 0].set_title("Stacked Area Plot of Mass Concentrations")
axes[0, 0].legend(loc="upper left")
axes[0, 0].yaxis.set_major_formatter(formatter)  # Apply scientific notation
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45)

# Second plot - Mass Absorption Efficiency over Time
axes[0, 1].plot(filtered_ds_psap_aux['Time'].squeeze(), Mass_absortion_eff, color="r", marker='.', linewidth=0)
axes[0, 1].set_xlabel("Datetime")
axes[0, 1].set_ylabel("Mass Absorption Efficiency (m²/g)")
axes[0, 1].yaxis.set_major_formatter(formatter)
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45)
axes[0, 1].grid()

# Third plot - Absorption Coefficient
axes[1, 0].plot(filtered_ds_psap_aux['Time'], filtered_ds_psap_aux['PSAP_LIN'], color="r", marker='.', linewidth=0)
axes[1, 0].set_xlabel("Time")
axes[1, 0].set_ylabel(r'Absorption Coefficient ($\mathrm{m^{-1}}$)')
axes[1, 0].yaxis.set_major_formatter(formatter)
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45)
axes[1, 0].grid()

# Fourth plot - Blue Scattering Coefficient
axes[1, 1].plot(ds_neph['datetime'], ds_neph['TSC_BLUU'])
axes[1, 1].set_xlabel("Time")
axes[1, 1].set_ylabel(r'Blue Scattering Coefficient ($\mathrm{m^{-1}}$)')
axes[1, 1].set_ylim(0, np.max(ds_neph['TSC_BLUU']) * 1.1)
axes[1, 1].yaxis.set_major_formatter(formatter)
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45)
axes[1, 1].grid()

# Adjust layout
plt.tight_layout()

# Show the final combined figure
plt.show()

In [ ]:
# Plot SO4 vs. datetime
plt.figure(figsize=(10, 5))
plt.plot(filtered_ds_psap_aux['Time'].squeeze(), Mass_absortion_eff, color="r", marker='.',linewidth=0)
#plt.xlim(filtered_ds_psap['Time'][550],filtered_ds_psap['Time'][3000])

# Formatting
plt.xlabel("Datetime")
plt.ylabel("Mass absortion efficiency m²/g")
#plt.title("Mass absortion efficiency Over Time")
plt.xticks(rotation=45)
plt.grid()

In [ ]:
# Step 5: Plot the resampled PSAP_LOG values
plt.figure(figsize=(10, 5))

# Plot resampled PSAP_LOG values
plt.plot(filtered_ds_psap_aux['Time'], filtered_ds_psap_aux['PSAP_LIN'], color="r", marker='.',linewidth=0)
#plt.plot(ds_test['Time'], ds_test['PSAP_LIN'], color="b", marker='o', linestyle='', markersize=4)
#plt.xlim(ds_test['Time'][200],ds_test['Time'][3500])
plt.xlabel('Time')
plt.ylabel(r'Absortion Coefficient ($\mathrm{m^{-1}}$)')
plt.xticks(rotation=45)
plt.grid()

# Tight layout for better plot formatting
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Step 5: Plot the resampled PSAP_LOG values
plt.figure(figsize=(10, 5))

# Plot resampled PSAP_LOG values
plt.plot(ds_neph['datetime'], ds_neph['TSC_BLUU'])

plt.xlabel('Time')
plt.ylabel(r'Blue Scattering Coefficient ($\mathrm{m^{-1}}$)')
plt.ylim(0,np.max(ds_neph['TSC_BLUU'])*1.1)
plt.xticks(rotation=45)
plt.grid()

# Tight layout for better plot formatting
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
df_aircraft = match_flight_with_blh(ds_psap,ds_bdl)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df_aircraft['time'], df_aircraft['height'], '.', label='Flight Altitude', alpha=0.6)
plt.plot(df_aircraft['time'], df_aircraft['blh'], '-', label='Boundary Layer Height (BLH)', linewidth=2)

# Step 3: Overlay event labels
for event_time, event_name in zip(df_flight["Start Time"], df_flight["Event"]):
    # Find the closest time in the dataset
    closest_idx = np.argmin(np.abs(ds_times - np.datetime64(event_time)))  # Index of nearest timestamp
    closest_time = df_aircraft['time'][closest_idx]
    height_at_time = df_aircraft['height'].values[closest_idx]  # Get height value

    # Plot event text
    plt.text(closest_time, height_at_time, event_name, fontsize=6, color="red", ha="left", va="bottom", rotation=45)


# Assuming df is your DataFrame and has columns "Start Time" and "Event"
start_up_time = df_flight[df_flight["Event"] == "Start-Up"]["Start Time"].iloc[0]

# Set xlim to start at 11:40
plt.xlim(start_up_time, df_aircraft['time'].values[-1])  # ds_times[-1] is the last time value


plt.ylabel("Height (m)")
plt.xlabel("Time")
plt.legend()
plt.title("Flight Altitude vs Boundary Layer Height (Hourly Reanalysis Match)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
def plot_flight_data_v4(flight_number,flight_date,
                     area_time, data_mass_concentration, 
                     filtered_ds_psap_aux, Mass_absortion_eff, 
                     filtered_ds_CO, 
                     ds_neph_datetime, ds_neph_TSC_BLUU,
                     ds_psap, df_flight, minlon, maxlon, minlat, maxlat):
    
    fig = plt.figure(figsize=(14, 15))

    # Formatter for scientific notation
    formatter = mticker.ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((-2, 2))


    # Step 1: Convert 'Start Time' from HHMMSS to datetime
    df_flight["Start Time"] = pd.to_datetime(df_flight["Start Time"], format="%H%M%S")  # Convert HHMMSS to time only
    df_flight["Start Time"] = df_flight["Start Time"].apply(lambda t: pd.Timestamp(flight_date) + pd.Timedelta(hours=t.hour, minutes=t.minute, seconds=t.second))
    
    start_up_time = df_flight[df_flight["Event"] == "Start-Up"]["Start Time"].iloc[0]
    # Define plot positions
    # Define updated plot positions for 5 subplots
    # Adjusted positions
    positions = {
        "stacked": [0.06, 0.74, 0.4, 0.2],         # Top-left
        "abs_coef": [0.06, 0.52, 0.4, 0.2],         # Middle-left
        "scatter_coef": [0.06, 0.30, 0.4, 0.2],        # Lower-left
        "height_time": [0.06, 0.08, 0.4, 0.2],    # Bottom-left
        "map": [0.52, 0.45, 0.42, 0.34],           # Top-right
        "co_conc": [0.52, 0.08, 0.4, 0.2],    # Bottom-right
    }


    # First plot - Stacked Area Plot
    ax1 = fig.add_axes(positions["stacked"])
    labels = ["BC", "SO4", "NH4", "ORG", "NO3", "Chl"]
    colors = ["black", "red", "orange", "green", "navy", "cyan"]
    ax1.stackplot(area_time, data_mass_concentration, labels=labels, colors=colors, alpha=0.7)
    #ax1.set_xlabel("Time")
    ax1.set_ylabel("Mass Concentration (µg/m³)")
    #print(area_time[0])
    ax1.set_title(f"Flight {flight_number} - Date: {flight_date}")
    ax1.legend(loc="upper left")
    ax1.yaxis.set_major_formatter(formatter)
    ax1.tick_params(axis='x', rotation=45)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    ax1.grid()
    ax1.tick_params(labelbottom=False)
    ax1.set_xlim(start_up_time, ds_psap['Time'].values[-1])

    # Second plot - Mass Absorption Efficiency over Time
    #ax2 = fig.add_axes(positions["abs_eff"])
    #ax2.plot(filtered_ds_psap_aux['Time'].squeeze(), Mass_absortion_eff, color="r", marker='.', linewidth=0)
    #ax2.set_xlabel("Datetime")
    #ax2.set_ylabel("Mass Absorption Efficiency (m²/g)")
    #ax2.set_title(f"Flight {flight_number} - Mass Absorption Efficiency\nDate: {flight_date}")
    #ax2.yaxis.set_major_formatter(formatter)
    #ax2.tick_params(axis='x', rotation=45)
    #ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    #ax2.grid()

    # Third plot - Absorption Coefficient
    ax2 = fig.add_axes(positions["abs_coef"])
    ax2.plot(filtered_ds_psap_aux['Time'], filtered_ds_psap_aux['PSAP_LIN'], color="r", marker='.', linewidth=0)
    #ax2.set_xlabel("Time")
    ax2.set_ylabel(r'Absorption Coefficient ($\mathrm{m^{-1}}$)')
    #ax2.set_title(f"Flight {flight_number} - Absorption Coefficient")
    ax2.yaxis.set_major_formatter(formatter)
    ax2.tick_params(axis='x', rotation=45)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    ax2.grid()
    ax2.tick_params(labelbottom=False)
    ax2.set_xlim(start_up_time, ds_psap['Time'].values[-1])

    # Fourth plot - Blue Scattering Coefficient
    ax4 = fig.add_axes(positions["scatter_coef"])
    ax4.plot(ds_neph_datetime, ds_neph_TSC_BLUU)
    #ax4.set_xlabel("Time")
    ax4.set_ylabel(r'Blue Scattering Coefficient ($\mathrm{m^{-1}}$)')
    #ax4.set_title(f"Flight {flight_number} - Blue Scattering Coefficient\nDate: {flight_date}")
    ax4.set_ylim(0, np.max(ds_neph_TSC_BLUU) * 1.1)
    ax4.yaxis.set_major_formatter(formatter)
    ax4.tick_params(axis='x', rotation=45)
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    ax4.grid()
    ax4.tick_params(labelbottom=False)
    ax4.set_xlim(start_up_time, ds_psap['Time'].values[-1])

    # Fifth plot - Flight Track Map (Cartopy)
    ax5 = fig.add_axes(positions["map"], projection=ccrs.PlateCarree())
    #ax5.set_extent([minlon, maxlon, minlat, maxlat], crs=ccrs.PlateCarree())
    ax5.coastlines()
    ax5.add_feature(cfeature.BORDERS, linestyle=':')
    ax5.add_feature(cfeature.LAND)
    ax5.add_feature(cfeature.OCEAN)
    # Add gridlines with only bottom and left labels
    gl = ax5.gridlines(draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False


    # Extract coordinates
    lat_valid = ds_psap['LAT_GIN']
    lon_valid = ds_psap['LON_GIN']
    
    # Filter out minimum values (e.g., invalid values like -999)
    valid_mask = (lat_valid < 0) & (lon_valid < -30)
    lat_filtered = lat_valid.where(valid_mask, drop=True)
    lon_filtered = lon_valid.where(valid_mask, drop=True)
    
    # Plot
    # ax5.set_extent([minlon, maxlon, minlat, maxlat], crs=ccrs.PlateCarree())
    ax5.set_extent([-66.5, -45.5, -12, -6], crs=ccrs.PlateCarree())
    ax5.scatter(lon_filtered, lat_filtered, s=1, transform=ccrs.PlateCarree())
    ax5.set_title(f"Flight {flight_number} - Flight Track")


    # Sixth plot - Height over Time with Event Labels
        # Sixth plot - Height over Time with Event Labels
    #valid_height_mask = ds_psap["HGT_RADR_FLAG"] == 0
    #ds_psap_hight_valid = ds_psap.where(valid_height_mask, drop=False)

    ax6 = fig.add_axes(positions["height_time"])
    #ax6.plot(ds_psap['Time'].values, ds_psap['HGT_RADR'].values, color="b", linestyle="-", alpha=0.7, label="Height Altimeter")
    ax6.plot(ds_psap['Time'].values, ds_psap['GPS_ALT'].values, color="g", linestyle="-", alpha=0.7, label="Height GPS")
    for event_time, event_name in zip(df_flight["Start Time"], df_flight["Event"]):
        closest_idx = np.argmin(np.abs(ds_psap['Time'].values - np.datetime64(event_time)))
        closest_time = ds_psap['Time'].values[closest_idx]
        height_at_time = ds_psap['GPS_ALT'].values[closest_idx][0]
        ax6.text(closest_time, height_at_time, event_name, fontsize=6, color="red", ha="left", va="bottom", rotation=45)
    
    #print(start_up_time)
    ax6.set_xlim(start_up_time, ds_psap['Time'].values[-1])
    ax6.set_xlabel("Time")
    ax6.set_ylabel("Height (m)")
    ax6.tick_params(axis='x', rotation=45)
    ax6.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    ax6.grid()
    #ax6.set_title(f"Flight {flight_number} - Height over Time\nDate: {flight_date}")


    # CO plot -CO concentration
    ax7 = fig.add_axes(positions["co_conc"])
    ax7.plot(filtered_ds_CO['Time'], filtered_ds_CO['CO_AERO'], color="r", marker='.', linewidth=0)
    ax2.set_xlabel("Time")
    ax7.set_ylabel(r'CO concentration ppb')
    #ax2.set_title(f"Flight {flight_number} - Absorption Coefficient")
    #ax7.yaxis.set_major_formatter(formatter)
    ax7.tick_params(axis='x', rotation=45)
    ax7.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    ax7.grid()
    #ax7.tick_params(labelbottom=False)
    ax7.set_xlim(start_up_time, ds_psap['Time'].values[-1])
    ax7.set_ylim(0,500)

    # Save and show the figure
    #plt.tight_layout()
    plt.savefig(f'flight_{flight_number}_v5.png', dpi=300)
    print(f"Figure saved as flight_{flight_number}_v5.png")
    #plt.show()

    return

In [ ]:
def plot_flight_data_v5(flight_number,flight_date,
                     area_time, data_mass_concentration, 
                     filtered_ds_psap_aux, Mass_absortion_eff, 
                     filtered_ds_CO, 
                     ds_neph_datetime, ds_neph_TSC_BLUU,
                     ds_psap, df_flight, minlon, maxlon, minlat, maxlat,
                     df_bdl):
    
    fig = plt.figure(figsize=(14, 10))

    # Formatter for scientific notation
    formatter = mticker.ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((-2, 2))


    # Step 1: Convert 'Start Time' from HHMMSS to datetime
    df_flight["Start Time"] = pd.to_datetime(df_flight["Start Time"], format="%H%M%S")  # Convert HHMMSS to time only
    df_flight["Start Time"] = df_flight["Start Time"].apply(lambda t: pd.Timestamp(flight_date) + pd.Timedelta(hours=t.hour, minutes=t.minute, seconds=t.second))
    
    start_up_time = df_flight[df_flight["Event"] == "Start-Up"]["Start Time"].iloc[0]
    # Define plot positions
    # Define updated plot positions for 5 subplots
    # Adjusted positions
    #positions = {
    #    "stacked": [0.06, 0.74, 0.4, 0.2],         # Top-left
    #    "abs_coef": [0.52, 0.74, 0.4, 0.2],         # Top-left
    #    "scatter_coef": [0.06, 0.30, 0.4, 0.2],        # Lower-left
    #    "height_time": [0.06, 0.52, 0.4, 0.2],    # Mid-left
    #    "map": [0.52, 0.20, 0.42, 0.34],           # Lower-right
    #    "co_conc": [0.52, 0.52, 0.4, 0.2],    # Mid-right
    #}
    positions = {
    "stacked":       [0.06, 0.70, 0.4, 0.25],  # Top-left
    "abs_coef":      [0.52, 0.70, 0.4, 0.25],  # Top-right
    "height_time":   [0.06, 0.40, 0.4, 0.25],  # Middle-left
    "co_conc":       [0.52, 0.40, 0.4, 0.25],  # Middle-right
    "scatter_coef":  [0.06, 0.10, 0.4, 0.25],  # Bottom-left
    "map":           [0.52, 0.08, 0.4, 0.25],  # Bottom-right
    }
    #[0.06, 0.52, 0.4, 0.2]
    #[0.06, 0.52, 0.4, 0.2]


    # First plot - Stacked Area Plot
    ax1 = fig.add_axes(positions["stacked"])
    labels = ["BC", "SO4", "NH4", "ORG", "NO3", "Chl"]
    colors = ["black", "red", "orange", "green", "navy", "cyan"]
    ax1.stackplot(area_time, data_mass_concentration, labels=labels, colors=colors, alpha=0.7)
    #ax1.set_xlabel("Time")
    ax1.set_ylabel("Mass Concentration (µg/m³)")
    #print(area_time[0])
    ax1.set_title(f"Flight {flight_number} - Date: {flight_date}")
    ax1.legend(loc="upper left")
    ax1.yaxis.set_major_formatter(formatter)
    ax1.tick_params(axis='x', rotation=45)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    ax1.grid()
    ax1.tick_params(labelbottom=False)
    ax1.set_xlim(start_up_time, ds_psap['Time'].values[-1])

    # Second plot - Mass Absorption Efficiency over Time
    #ax2 = fig.add_axes(positions["abs_eff"])
    #ax2.plot(filtered_ds_psap_aux['Time'].squeeze(), Mass_absortion_eff, color="r", marker='.', linewidth=0)
    #ax2.set_xlabel("Datetime")
    #ax2.set_ylabel("Mass Absorption Efficiency (m²/g)")
    #ax2.set_title(f"Flight {flight_number} - Mass Absorption Efficiency\nDate: {flight_date}")
    #ax2.yaxis.set_major_formatter(formatter)
    #ax2.tick_params(axis='x', rotation=45)
    #ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    #ax2.grid()

    # Third plot - Absorption Coefficient
    ax2 = fig.add_axes(positions["abs_coef"])
    ax2.plot(filtered_ds_psap_aux['Time'], filtered_ds_psap_aux['PSAP_LIN'], color="r", marker='.', linewidth=0)
    #ax2.set_xlabel("Time")
    ax2.set_ylabel(r'Absorption Coefficient ($\mathrm{m^{-1}}$)')
    #ax2.set_title(f"Flight {flight_number} - Absorption Coefficient")
    ax2.yaxis.set_major_formatter(formatter)
    ax2.tick_params(axis='x', rotation=45)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    ax2.grid()
    ax2.tick_params(labelbottom=False)
    ax2.set_xlim(start_up_time, ds_psap['Time'].values[-1])

    # Fourth plot - Blue Scattering Coefficient
    ax4 = fig.add_axes(positions["scatter_coef"])
    ax4.plot(ds_neph_datetime, ds_neph_TSC_BLUU)
    ax4.set_xlabel("Time")
    ax4.set_ylabel(r'Blue Scattering Coefficient ($\mathrm{m^{-1}}$)')
    #ax4.set_title(f"Flight {flight_number} - Blue Scattering Coefficient\nDate: {flight_date}")
    ax4.set_ylim(0, np.max(ds_neph_TSC_BLUU) * 1.1)
    ax4.yaxis.set_major_formatter(formatter)
    ax4.tick_params(axis='x', rotation=45)
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    ax4.grid()
    #ax4.tick_params(labelbottom=True)
    ax4.set_xlim(start_up_time, ds_psap['Time'].values[-1])

    # Fifth plot - Flight Track Map (Cartopy)
    ax5 = fig.add_axes(positions["map"], projection=ccrs.PlateCarree())
    #ax5.set_extent([minlon, maxlon, minlat, maxlat], crs=ccrs.PlateCarree())
    ax5.coastlines()
    ax5.add_feature(cfeature.BORDERS, linestyle=':')
    ax5.add_feature(cfeature.LAND)
    ax5.add_feature(cfeature.OCEAN)
    # Add gridlines with only bottom and left labels
    gl = ax5.gridlines(draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False


    # Extract coordinates
    lat_valid = ds_psap['LAT_GIN']
    lon_valid = ds_psap['LON_GIN']
    
    # Filter out minimum values (e.g., invalid values like -999)
    valid_mask = (lat_valid < 0) & (lon_valid < -30)
    lat_filtered = lat_valid.where(valid_mask, drop=True)
    lon_filtered = lon_valid.where(valid_mask, drop=True)
    
    # Plot
    # ax5.set_extent([minlon, maxlon, minlat, maxlat], crs=ccrs.PlateCarree())
    ax5.set_extent([-66.5, -45.5, -12, -6], crs=ccrs.PlateCarree())
    ax5.scatter(lon_filtered, lat_filtered, s=1, transform=ccrs.PlateCarree())
    ax5.set_title(f"Flight {flight_number} - Flight Track")


    # Sixth plot - Height over Time with Event Labels and Boundary layer
    ax6 = fig.add_axes(positions["height_time"])

    ax6.plot(df_bdl['time'], df_bdl['height'], '.')
    ax6.plot(df_bdl['time'], df_bdl['blh'], '-', label='Boundary Layer',alpha=0.6,linewidth=2)
    
    # Step 3: Overlay event labels
    for event_time, event_name in zip(df_flight["Start Time"], df_flight["Event"]):
        # Find the closest time in the dataset
        closest_idx = np.argmin(np.abs(df_bdl['time'].values - np.datetime64(event_time)))  # Index of nearest timestamp
        closest_time = df_bdl['time'][closest_idx]
        height_at_time = df_bdl['height'].values[closest_idx]  # Get height value
    
        # Plot event text
        ax6.text(closest_time, height_at_time, event_name, fontsize=6, color="red", ha="left", va="bottom", rotation=45)
    
    
    # Assuming df is your DataFrame and has columns "Start Time" and "Event"
    start_up_time = df_flight[df_flight["Event"] == "Start-Up"]["Start Time"].iloc[0]
    
    # Set xlim to start at 11:40
    ax6.grid()
    ax6.set_ylabel("Height (m)")
    #ax6.set_xlabel("Time")
    ax6.tick_params(axis='x', rotation=45)
    ax6.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    ax6.legend(loc="upper right")
    ax6.tick_params(labelbottom=False)
    ax6.set_xlim(start_up_time, df_bdl['time'].values[-1])  # ds_times[-1] is the last time value


    # CO plot -CO concentration
    ax7 = fig.add_axes(positions["co_conc"])
    ax7.plot(filtered_ds_CO['Time'], filtered_ds_CO['CO_AERO'], color="r", marker='.', linewidth=0)
    ax7.set_xlabel("Time")
    ax7.set_ylabel(r'CO concentration ppb')
    #ax2.set_title(f"Flight {flight_number} - Absorption Coefficient")
    #ax7.yaxis.set_major_formatter(formatter)
    ax7.tick_params(axis='x', rotation=45)
    ax7.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format as hours:minutes
    ax7.grid()
    #ax7.tick_params(labelbottom=False)
    ax7.set_xlim(start_up_time, ds_psap['Time'].values[-1])
    ax7.set_ylim(0,1000)

    # Save and show the figure
    #plt.tight_layout()
    plt.savefig(f'flight_{flight_number}_v5.png', dpi=300)
    print(f"Figure saved as flight_{flight_number}_v5.png")
    #plt.show()

    return

In [ ]:
ds_psap

In [ ]:
# List to store results
results = []



# Iterate over all matching flights
for sp2_file, ams_file, nc_file, neph_file, flight_comments_file in zip(filtered_sp2, filtered_ams, filtered_nc, filtered_neph,filtered_sum):

    #print(neph_file)
    ds_sp2,ds_ams,ds_neph = dataset_sp2_ams_neph(sp2_file,ams_file,neph_file)
    ds_psap = xr.open_dataset(nc_file)

    #Calculate absortion efficiency 
    ds_test,tm_test=calculate_absortion_eff(ds_sp2,ds_ams,ds_psap)

    #Extract concentration data
    area_time,data_mass_concentration = Area_plot(ds_sp2,ds_ams,ds_psap)

    #Filter_CO_data
    valid_CO = ds_psap["CO_AERO_FLAG"] == 0
    filtered_CO = ds_psap.where(valid_CO, drop=True)

    #Filter absortion data
    valid_PSAP_mask = ds_test["PSAP_LIN_FLAG"] == 0
    filtered_ds_psap = ds_test.where(valid_PSAP_mask, drop=False)
    filtered_ds_psap_aux = ds_test.where(valid_PSAP_mask, drop=True)
    # Ensure both arrays are 1D and aligned
    psap_values = filtered_ds_psap["PSAP_LIN"].values.squeeze()
    tm_values = tm_test.squeeze()
    
    # Perform element-wise division, ignoring NaNs
    result = np.divide(psap_values, tm_values*1e-6)
    
    # Drop NaN values from the result if needed
    Mass_absortion_eff = result.where(~np.isnan(result), drop=True)

    # Compute the mean, handling cases where the list might be empty
    mean_mass_absorption_eff_g = np.nanmean(Mass_absortion_eff) 
    std_mass_absorption_eff_g = np.nanstd(Mass_absortion_eff) 
    mean_std = std_mass_absorption_eff_g/np.sqrt(len(Mass_absortion_eff))
    

    # Extract the flight number (b###) from the file path
    match = re.search(r'b\d{3}', nc_file)  # Looks for 'b' followed by 3 digits
    flight_number = match.group(0) if match else "Unknown"
    # Extract the date in yyyymmdd format
    date_match = re.search(r'(\d{8})', nc_file)
    if date_match:
        date_str = date_match.group(1)
        flight_date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    else:
        flight_date = "Unknown"
    
    #print(flight_date)
    #Flight comments dataset
    df_flight = flight_comments_df(flight_comments_file)

    df_aircraft = match_flight_with_blh(ds_psap,ds_bdl)

    plot_flight_data_v5( 
        flight_number=flight_number,
        flight_date = flight_date,
        area_time=area_time,
        data_mass_concentration=data_mass_concentration,
        filtered_ds_psap_aux=filtered_ds_psap_aux,
        Mass_absortion_eff=Mass_absortion_eff,
        filtered_ds_CO=filtered_CO,
        ds_neph_datetime=ds_neph['datetime'],
        ds_neph_TSC_BLUU=ds_neph['TSC_BLUU'],
        ds_psap=ds_psap, 
        df_flight=df_flight, 
        minlon = minlon, 
        maxlon = maxlon,
        minlat = minlat, 
        maxlat = maxlat,
        df_bdl = df_aircraft)
    

    # Store results in a list
    results.append({
        "flight": flight_number,
        "mean_mass_absorption_eff_g": mean_mass_absorption_eff_g,
        "std_mass_absorption_eff_g": std_mass_absorption_eff_g,
        "mean_std": mean_std,
        "N": len(Mass_absortion_eff)
    })